# M4: data layer

Records with provenance, then ExoMol and MARVEL readers, network inversion, the error prior, round-trip validation and the store.

* If `DATA_DIR` holds files fetched with `scripts/fetch_data.py` (for example an attached Kaggle dataset), the notebook reads them.
* Otherwise it runs on a **synthetic demonstration** generated from the library's own model, so every step still executes. Nothing in that demonstration is real data.

See `docs/data_sources.md` for the recommended datasets (H₂CO MARVEL first).

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import riss_drucker as rd
from riss_drucker.data import exomol, marvel, network, roundtrip, store, error_prior

WORK = Path(os.environ.get("RISS_DRUCKER_WORK", "/kaggle/working" if Path("/kaggle/working").is_dir() else "."))
DATA_DIR = Path(os.environ.get("RISS_DRUCKER_DATA", "/kaggle/input/riss-drucker-data"))
real = sorted(DATA_DIR.glob("*.states.bz2")) if DATA_DIR.is_dir() else []
print("real ExoMol states files:", [p.name for p in real] or "none -> synthetic demonstration")

In [ ]:
if real:
    stem = real[0].name.replace(".states.bz2", "")
    fields = exomol.read_def(DATA_DIR / f"{stem}.def.json")["states_fields"]
    states = exomol.read_states(real[0], fields, j_max=20, e_max=5000.0)
    print(stem, len(states["i"]), "states with J <= 20, E <= 5000 cm-1; fields:", fields)
else:
    print("skipped: no real files")

## Network inversion and round trip (synthetic demonstration when no real MARVEL file is present)

In [ ]:
TRUE = {"A": 9.405, "B": 1.2953, "C": 1.1342, "DeltaJ": 2.4e-6, "DeltaJK": 9.7e-5, "DeltaK": 4.4e-3,
        "deltaJ": 3.4e-7, "deltaK": 2.7e-5}   # illustrative, not H2CO's published constants
m = rd.compile_asym_rotor(j_max=10, order=4)
sol = rd.solve(m, TRUE, nlowest=21, keep_vectors=True, method="dense")
lines = rd.spectra.line_strengths(m, sol, (1.0, 0.0, 0.0)) + rd.spectra.line_strengths(m, sol, (0.0, 1.0, 0.0))
rng = np.random.default_rng(0)
qn = lambda lab: (lab.split("(")[0], *lab.split("(")[1].rstrip(")").split(","))
trans = [marvel.MarvelTransition(l.wavenumber + 1e-4 * rng.standard_normal(), 1e-4, qn(l.upper_label), qn(l.lower_label), f"demo.{i}")
         for i, l in enumerate(lines)]
for i in (5, 50, 120):
    trans[i].wavenumber += 3e-3          # injected bad lines
res = network.invert(trans, anchors={("0", "0", "0"): 0.0})
print(f"{len(res.levels)} levels, {len(res.components)} component(s), outliers flagged: {res.outliers}")

In [ ]:
MHZ = rd.data.records.MHZ_PER_CM
name_map = {"A": "A", "B": "B", "C": "C", "DJ": "DeltaJ", "DJK": "DeltaJK", "DK": "DeltaK", "dJ": "deltaJ", "dK": "deltaK"}
inv = {v: k for k, v in name_map.items()}
cite = {"citation": "synthetic demonstration", "extraction_method": "manual"}
const = {"schema_version": 1, "kind": "constants", "molecule": {"formula": "demo"}, "source": "measured",
         "provenance": cite, "units": {"energy": "MHz"}, "uncertainty_definition": "1sigma",
         "constants": {inv[k]: {"value": v * MHZ} for k, v in TRUE.items()},
         "convention": {"model": "asym_rotor", "reduction": "A", "representation": "Ir", "name_map": name_map},
         "model_spec": rd.ModelSpec.asym_rotor(j_max=10, order=4).canonical(), "fit_statistics": {"rms": 0.05}}
refs = [{"schema_version": 1, "kind": "transition", "molecule": {"formula": "demo"}, "source": "measured",
         "provenance": cite, "units": {"energy": "MHz"}, "uncertainty_definition": "1sigma",
         "value": l.wavenumber * MHZ + 0.05 * rng.standard_normal(), "uncertainty": 0.05,
         "upper": {"label": l.upper_label}, "lower": {"label": l.lower_label}} for l in lines[:200]]
ok = roundtrip.validate_constants(const, refs, model=m)
bad = json.loads(json.dumps(const)); bad["constants"]["dK"]["value"] *= -1
print("round trip, as extracted:", ok["validation"]["status"], "| with a sign flip:",
      roundtrip.validate_constants(bad, refs, model=m)["validation"]["status"])
st = store.RecordStore(WORK / "record_store_demo")
for r in refs:
    st.add(r)
obs = st.fit_observations(m, formula="demo")
fit = rd.fit(m, obs, rd.synth.perturb_theta(TRUE, 1e-4, seed=1))
print(f"fit from the store: {len(obs)} lines, chi2/dof = {fit.reduced_chi2:.2f}, certified = {fit.certified}")